# Gradient Boosting и Multi-label классификация в CatBoost

**Gradient Boosting** — это ансамбль **Деревьев решений**, обучающихся последовательно с помощью градиентного спуска.

В библиотеках **CatBoost / LightGBM** для работы с задачами multi-label (многометочная классификация) используется стратегия: **одно дерево = один таргет**.

## 📊 Пример структуры данных

Таблица с признаками пациентов и множественными диагнозами:

| Осмотр_ID | Селект_1 (Кашель) | Селект_2 (Горло) | ... | Селект_30 (Хрипы) | Острый бронхит | Гипертония | Сахарный диабет | ... | Диагноз_100 |
|---|---|---|---|---|---|---|---|---|---|
| **Строка 1** | Влажный | Красное | ... | Да | **1** | 0 | 0 | ... | 0 |
| **Строка 2** | Нет кашля | Норма | ... | Нет | 0 | 0 | 0 | ... | **1** |
| **Строка 3** | Сухой | Красное | ... | Да | 0 | **1** | **1** | ... | 0 |

CatBoost не может работать с таблицей, где всё свалено в кучу. Ему нужно четко указать: **где признаки (загадки)**, а **где правильные ответы (отгадки)**.

Поэтому в коде на Python мы мысленно «разрезаем» эту таблицу по вертикали на две части:

1. **Матрица признаков (`X`):** Прямоугольник размером **100 000 строк на 30 колонок** (только селекты).
2. **Матрица ответов (`Y`):** Прямоугольник размером **100 000 строк на 100 колонок** (нули и единицы диагнозов).

## ⚙️ Как CatBoost начинает работу с данными

Когда вы запускаете команду `model.fit(X, Y)`, CatBoost реализует стратегию **«Одно дерево — один диагноз»**:

1. **Запуск параллельных процессов:** По одному процессу на каждую колонку из матрицы `Y` (всего 100 процессов).
2. **Процесс №1 (для Острого бронхита):**
   * Берет всю матрицу селектов `X` и **только первую колонку** из матрицы `Y` (`Острый бронхит`).
   * Применяет **Target Encoding** к 30 селектам (переводит текст в числа на основе частоты единиц в колонке бронхита).
   * С помощью **градиентного спуска** строит первое **дерево решений**.
3. **Процесс №2 (для Гипертонии):**
   * Берет ту же матрицу селектов `X`, но смотрит **только на вторую колонку** (`Гипертония`).
   * Делает свой независимый Target Encoding (учитывая специфичные для гипертонии признаки).
   * Строит свою команду деревьев решений для угадывания единичек в колонке гипертонии.
4. Процесс повторяется для всех 100 колонок-диагнозов.

## 🏁 Финал обучения

На выходе CatBoost выдает **один обученный файл модели**, внутри которого содержатся 100 изолированных команд деревьев решений.

**Предсказание для нового пациента:**
Когда поступает новый пациент (только с 30 селектами), модель одновременно прогоняет его данные через все 100 команд деревьев и выдает массив из 100 вероятностей (например, `[0.92, 0.01, 0.85, ... 0.00]`). Интерфейс сортирует этот массив и подсвечивает врачу наиболее вероятные диагнозы.